# Finlora Fraud Risk-Scoring: Data Cleaning

First deliverable in the pipeline: load the raw transaction and account data, merge them, identify missing values and duplicate records, clean the data, and export a single cleaned/merged dataset for downstream notebooks (EDA, modeling) to consume.

**Contents**
1. Load data
2. Merge transactions with accounts
3. Identify missing values
4. Identify duplicate records
5. Clean the data
6. Save cleaned dataset

In [1]:
import pandas as pd

pd.set_option("display.max_columns", None)

## 1. Load data

In [2]:
transactions = pd.read_csv("../Data/finlora_transactions.csv")
accounts = pd.read_csv("../Data/finlora_accounts.csv")

print("transactions:", transactions.shape)
print("accounts:", accounts.shape)

transactions.head()

transactions: (126000, 24)
accounts: (7200, 8)


,transaction_id,account_id,account_type,kyc_tier,timestamp,day_of_week,hour_of_day,description,merchant_name,merchant_category,channel,amount,currency,amount_to_avg_ratio,avg_transaction_amount_30d,transaction_velocity_1h,transaction_country,home_country,is_cross_border,device_id,is_new_device,account_age_days,status,is_fraud
0,FLR250216186265,FLR-ACC-100000,Individual,Tier3_Enhanced,2025-02-16 14:55:35,Sunday,14,CNP PURCHASE - BOLT,Bolt,Travel,Card Not Present,84233.00,NGN,9.21,9143.33,0,NG,NG,0,NaN,NaN,903,Completed,0
1,FLR250423143305,FLR-ACC-100000,Individual,Tier3_Enhanced,2025-04-23 13:35:40,Wednesday,13,WEB PURCHASE - SLACK,Slack,Subscription/SaaS,Web Dashboard,9143.33,NGN,1.00,9143.33,0,NG,NG,0,NaN,NaN,969,Declined,0
2,FLR250424154897,FLR-ACC-100000,Individual,Tier3_Enhanced,2025-04-24 12:30:34,Thursday,12,MOBILE PURCHASE - JUSTRITE SUPERSTORE,Justrite Superstore,Groceries,Mobile App,20639.00,NGN,2.26,9143.33,0,NG,NG,0,DEV-9055235108,0.0,970,Completed,0
3,FLR250428101772,FLR-ACC-100000,Individual,Tier3_Enhanced,2025-04-28 14:41:59,Monday,14,CNP PURCHASE - ADOBE CREATIVE CLOUD,Adobe Creative Cloud,Subscription/SaaS,Card Not Present,3177.43,NGN,0.21,14891.16,0,NG,NG,0,NaN,NaN,974,Completed,0
4,FLR250628102827,FLR-ACC-100000,Individual,Tier3_Enhanced,2025-06-28 16:15:47,Saturday,16,MOBILE PURCHASE - APPLE STORE,Apple Store,Electronics,Mobile App,97839.52,NGN,1.97,49609.59,0,NG,NG,0,DEV-9055235108,0.0,1035,Declined,0


In [3]:
accounts.head()

,account_id,account_holder_name,account_type,home_country,currency,kyc_tier,account_created_date,personal_spend_baseline_usd
0,FLR-ACC-100000,Amaka Okafor,Individual,NG,NGN,Tier3_Enhanced,2022-08-28,63.23
1,FLR-ACC-100001,Bluewave Foods Ltd,Business,NG,NGN,Tier1_Basic,2026-04-27,810.68
2,FLR-ACC-100002,Global Trading LLC,Business,US,USD,Tier2_Verified,2026-03-29,433.67
3,FLR-ACC-100003,Femi Adeyemi,Individual,NG,NGN,Tier3_Enhanced,2025-09-15,19.18
4,FLR-ACC-100004,Ngozi Brown,Individual,NG,NGN,Tier3_Enhanced,2024-06-18,28.68


## 2. Merge transactions with accounts

Both files carry `account_type`, `kyc_tier`, `home_country`, and `currency`; after the merge we keep a single, non-suffixed copy of each (from the transactions table, since that's the record actually in effect at transaction time).

In [4]:
shared_cols = ["account_type", "kyc_tier", "home_country", "currency"]
accounts_for_merge = accounts.drop(columns=shared_cols)

df = transactions.merge(accounts_for_merge, on="account_id", how="left")
print("merged:", df.shape)
df.head()

merged: (126000, 27)


,transaction_id,account_id,account_type,kyc_tier,timestamp,day_of_week,hour_of_day,description,merchant_name,merchant_category,channel,amount,currency,amount_to_avg_ratio,avg_transaction_amount_30d,transaction_velocity_1h,transaction_country,home_country,is_cross_border,device_id,is_new_device,account_age_days,status,is_fraud,account_holder_name,account_created_date,personal_spend_baseline_usd
0,FLR250216186265,FLR-ACC-100000,Individual,Tier3_Enhanced,2025-02-16 14:55:35,Sunday,14,CNP PURCHASE - BOLT,Bolt,Travel,Card Not Present,84233.00,NGN,9.21,9143.33,0,NG,NG,0,NaN,NaN,903,Completed,0,Amaka Okafor,2022-08-28,63.23
1,FLR250423143305,FLR-ACC-100000,Individual,Tier3_Enhanced,2025-04-23 13:35:40,Wednesday,13,WEB PURCHASE - SLACK,Slack,Subscription/SaaS,Web Dashboard,9143.33,NGN,1.00,9143.33,0,NG,NG,0,NaN,NaN,969,Declined,0,Amaka Okafor,2022-08-28,63.23
2,FLR250424154897,FLR-ACC-100000,Individual,Tier3_Enhanced,2025-04-24 12:30:34,Thursday,12,MOBILE PURCHASE - JUSTRITE SUPERSTORE,Justrite Superstore,Groceries,Mobile App,20639.00,NGN,2.26,9143.33,0,NG,NG,0,DEV-9055235108,0.0,970,Completed,0,Amaka Okafor,2022-08-28,63.23
3,FLR250428101772,FLR-ACC-100000,Individual,Tier3_Enhanced,2025-04-28 14:41:59,Monday,14,CNP PURCHASE - ADOBE CREATIVE CLOUD,Adobe Creative Cloud,Subscription/SaaS,Card Not Present,3177.43,NGN,0.21,14891.16,0,NG,NG,0,NaN,NaN,974,Completed,0,Amaka Okafor,2022-08-28,63.23
4,FLR250628102827,FLR-ACC-100000,Individual,Tier3_Enhanced,2025-06-28 16:15:47,Saturday,16,MOBILE PURCHASE - APPLE STORE,Apple Store,Electronics,Mobile App,97839.52,NGN,1.97,49609.59,0,NG,NG,0,DEV-9055235108,0.0,1035,Declined,0,Amaka Okafor,2022-08-28,63.23


## 3. Identify missing values

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 126000 entries, 0 to 125999
Data columns (total 27 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   transaction_id               126000 non-null  str    
 1   account_id                   126000 non-null  str    
 2   account_type                 126000 non-null  str    
 3   kyc_tier                     126000 non-null  str    
 4   timestamp                    126000 non-null  str    
 5   day_of_week                  126000 non-null  str    
 6   hour_of_day                  126000 non-null  int64  
 7   description                  126000 non-null  str    
 8   merchant_name                97390 non-null   str    
 9   merchant_category            126000 non-null  str    
 10  channel                      126000 non-null  str    
 11  amount                       126000 non-null  float64
 12  currency                     126000 non-null  str    
 13  amount_to_

In [6]:
missing = df.isna().sum()
missing = missing[missing > 0].sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(2)
pd.DataFrame({"missing_count": missing, "missing_pct": missing_pct})

,missing_count,missing_pct
merchant_name,28610,22.71
device_id,14334,11.38
is_new_device,14334,11.38


**Findings:**
- `merchant_name` is missing for ~22.7% of rows -- these are transaction types without a merchant counterparty (ATM withdrawals, transfers, etc.), not a data error.
- `device_id` and `is_new_device` are both missing for the exact same ~11.4% of rows -- a structural gap concentrated in channels that don't carry a device context (e.g. USSD, API/Integration), not random/independent missingness.

In [7]:
same_rows_missing = (df["device_id"].isna() == df["is_new_device"].isna()).all()
print("device_id and is_new_device missing on exactly the same rows:", same_rows_missing)

df.loc[df["device_id"].isna(), "channel"].value_counts()

device_id and is_new_device missing on exactly the same rows: True


channel
Mobile App          4761
Web Dashboard       3636
Card Not Present    1925
Card Present        1690
API/Integration     1470
USSD                 852
Name: count, dtype: int64

## 4. Identify duplicate records

In [8]:
full_dupes = df.duplicated().sum()
id_dupes = df["transaction_id"].duplicated().sum()

print(f"Fully duplicated rows: {full_dupes}")
print(f"Duplicate transaction_id values: {id_dupes}")
print(f"Unique transaction_id count: {df['transaction_id'].nunique()} of {len(df)} rows")

Fully duplicated rows: 0
Duplicate transaction_id values: 0
Unique transaction_id count: 126000 of 126000 rows


**Finding:** no fully duplicated rows and no repeated `transaction_id` values -- every transaction is a distinct record, so no deduplication is needed.

## 5. Clean the data

- Missing `merchant_name` is filled with `"Unknown"` rather than dropped -- the row itself is a valid transaction, it just has no merchant.
- Missing `device_id` / `is_new_device` are filled with an explicit `"NoDevice"` / `-1` category rather than dropped, which would otherwise remove a disproportionate share of the already-rare fraud-positive rows.
- `is_new_device`, `is_cross_border`, and `is_fraud` are cast to `int` (they load as floats/object once `is_new_device` carries NaNs and `-1` sentinels).

In [9]:
df_clean = df.copy()

df_clean["merchant_name"] = df_clean["merchant_name"].fillna("Unknown")
df_clean["device_id"] = df_clean["device_id"].fillna("NoDevice")
df_clean["is_new_device"] = df_clean["is_new_device"].fillna(-1).astype(int)
df_clean["is_cross_border"] = df_clean["is_cross_border"].astype(int)
df_clean["is_fraud"] = df_clean["is_fraud"].astype(int)

df_clean.isna().sum().sum()

np.int64(0)

**Data quality check:** `amount` and `avg_transaction_amount_30d` are transaction-amount fields, so they shouldn't be negative. These rows are flagged rather than dropped or altered -- the sampled negative-amount rows are all tagged `merchant_category = Retail`, consistent with refunds/returns encoded as negative amounts rather than a data error, and dropping them would silently discard real transactions.

In [10]:
for col in ["amount", "avg_transaction_amount_30d"]:
    neg = (df_clean[col] < 0).sum()
    zero = (df_clean[col] == 0).sum()
    print(f"{col}: {neg} negative, {zero} zero, of {len(df_clean):,} rows")

df_clean["has_invalid_amount"] = (df_clean["amount"] < 0) | (df_clean["avg_transaction_amount_30d"] < 0)
print(f"\nRows flagged has_invalid_amount: {df_clean['has_invalid_amount'].sum()}")

amount: 291 negative, 3 zero, of 126,000 rows
avg_transaction_amount_30d: 144 negative, 3 zero, of 126,000 rows

Rows flagged has_invalid_amount: 400


In [11]:
print("Final shape:", df_clean.shape)
print("Remaining missing values:", df_clean.isna().sum().sum())
print("Remaining duplicate rows:", df_clean.duplicated().sum())
df_clean.head()

Final shape: (126000, 28)
Remaining missing values: 0


Remaining duplicate rows: 0


,transaction_id,account_id,account_type,kyc_tier,timestamp,day_of_week,hour_of_day,description,merchant_name,merchant_category,channel,amount,currency,amount_to_avg_ratio,avg_transaction_amount_30d,transaction_velocity_1h,transaction_country,home_country,is_cross_border,device_id,is_new_device,account_age_days,status,is_fraud,account_holder_name,account_created_date,personal_spend_baseline_usd,has_invalid_amount
0,FLR250216186265,FLR-ACC-100000,Individual,Tier3_Enhanced,2025-02-16 14:55:35,Sunday,14,CNP PURCHASE - BOLT,Bolt,Travel,Card Not Present,84233.00,NGN,9.21,9143.33,0,NG,NG,0,NoDevice,-1,903,Completed,0,Amaka Okafor,2022-08-28,63.23,False
1,FLR250423143305,FLR-ACC-100000,Individual,Tier3_Enhanced,2025-04-23 13:35:40,Wednesday,13,WEB PURCHASE - SLACK,Slack,Subscription/SaaS,Web Dashboard,9143.33,NGN,1.00,9143.33,0,NG,NG,0,NoDevice,-1,969,Declined,0,Amaka Okafor,2022-08-28,63.23,False
2,FLR250424154897,FLR-ACC-100000,Individual,Tier3_Enhanced,2025-04-24 12:30:34,Thursday,12,MOBILE PURCHASE - JUSTRITE SUPERSTORE,Justrite Superstore,Groceries,Mobile App,20639.00,NGN,2.26,9143.33,0,NG,NG,0,DEV-9055235108,0,970,Completed,0,Amaka Okafor,2022-08-28,63.23,False
3,FLR250428101772,FLR-ACC-100000,Individual,Tier3_Enhanced,2025-04-28 14:41:59,Monday,14,CNP PURCHASE - ADOBE CREATIVE CLOUD,Adobe Creative Cloud,Subscription/SaaS,Card Not Present,3177.43,NGN,0.21,14891.16,0,NG,NG,0,NoDevice,-1,974,Completed,0,Amaka Okafor,2022-08-28,63.23,False
4,FLR250628102827,FLR-ACC-100000,Individual,Tier3_Enhanced,2025-06-28 16:15:47,Saturday,16,MOBILE PURCHASE - APPLE STORE,Apple Store,Electronics,Mobile App,97839.52,NGN,1.97,49609.59,0,NG,NG,0,DEV-9055235108,0,1035,Declined,0,Amaka Okafor,2022-08-28,63.23,False


## 6. Save cleaned dataset

In [12]:
OUT_PATH = "../Data/finlora_cleaned_merged.csv"
df_clean.to_csv(OUT_PATH, index=False)
print("Saved:", OUT_PATH)

Saved: ../Data/finlora_cleaned_merged.csv
